In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

import os

print(os.getcwd())

E:\Datos_User\Desktop\UNIVERSIDAD\Especializacion Ingenieria de Software\SegundoSemestre\Nueva carpeta\proyectoaulaNE


In [2]:
DATASET_DIR = Path("Dataset_proyecto_aula")

archivos = sorted(DATASET_DIR.glob("*.csv"))

print("Archivos encontrados:", len(archivos))

SEGMENTOS = [
    "Estrato 1",
    "Estrato 2",
    "Estrato 3",
    "Estrato 4",
    "Estrato 5",
    "Estrato 6",
    "Total Residencial",
    "Industrial",
    "Comercial",
    "Oficial",
    "Otros",
    "Total No Residencial"
]

Archivos encontrados: 15


In [3]:
def obtener_periodo(nombre_archivo):
    anio, mes = nombre_archivo.replace(".csv", "").split("_")

    return int(anio), int(mes)

In [4]:
def limpiar_empresa(nombre):
    if pd.isna(nombre):
        return np.nan

    return (
        str(nombre)
        .upper()
        .strip()
    )

In [5]:
def transformar_reporte(df, anio, mes):
    # Factura promedio únicamente
    df = df[
        df["Variable Calculada"]
        .astype(str)
        .str.lower()
        .str.contains("factura promedio", na=False)
    ]

    # Empresa válida
    df = df.dropna(subset=["Empresa"])

    # Normalizar nombres
    df["Empresa"] = df["Empresa"].apply(limpiar_empresa)

    # Variables temporales
    df["Anio"] = anio
    df["Mes"] = mes
    df["Periodo"] = f"{anio}-{mes:02d}"

    # Conversión numérica
    for col in SEGMENTOS:

        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    # Formato largo
    df = df.melt(
        id_vars=[
            "Empresa",
            "Anio",
            "Mes",
            "Periodo"
        ],
        value_vars=SEGMENTOS,
        var_name="Segmento",
        value_name="Factura_Promedio_COP"
    )

    # Eliminar nulos
    df = df.dropna(
        subset=["Factura_Promedio_COP"]
    )

    return df


In [6]:
dfs = []

for archivo in sorted(DATASET_DIR.glob("*.csv")):
    print(f"Procesando {archivo.name}")

    anio, mes = obtener_periodo(archivo.name)

    df = pd.read_csv(
        archivo,
        encoding="latin-1",
        na_values=[
            "ND",
            "N/D",
            "-",
            ""
        ]
    )

    df = transformar_reporte(
        df,
        anio,
        mes
    )

    dfs.append(df)
    df_final = pd.concat(
    dfs,
    ignore_index=True
)

Procesando 2025_01.csv
Procesando 2025_02.csv
Procesando 2025_03.csv
Procesando 2025_04.csv
Procesando 2025_05.csv
Procesando 2025_06.csv
Procesando 2025_07.csv
Procesando 2025_08.csv
Procesando 2025_09.csv
Procesando 2025_10.csv
Procesando 2025_11.csv
Procesando 2025_12.csv
Procesando 2026_01.csv
Procesando 2026_02.csv
Procesando 2026_03.csv


In [7]:
df_final.head()

,Empresa,Anio,Mes,Periodo,Segmento,Factura_Promedio_COP
0,AIR-E S.A.S. E.S.P.,2025,1,2025-01,Estrato 1,273094.53
1,CARIBEMAR DE LA COSTA S.A.S. E.S.P.,2025,1,2025-01,Estrato 1,200250.84
2,CELSIA COLOMBIA S.A. E.S.P.,2025,1,2025-01,Estrato 1,87115.06
3,CENTRALES ELECTRICAS DEL NORTE DE SANTANDER S....,2025,1,2025-01,Estrato 1,119172.97
4,CENTRALES ELECTRICAS DE NARIÑO S.A. E.S.P.,2025,1,2025-01,Estrato 1,72510.58


In [8]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 5809 entries, 0 to 5808
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Empresa               5809 non-null   str    
 1   Anio                  5809 non-null   int64  
 2   Mes                   5809 non-null   int64  
 3   Periodo               5809 non-null   str    
 4   Segmento              5809 non-null   str    
 5   Factura_Promedio_COP  5809 non-null   float64
dtypes: float64(1), int64(2), str(3)
memory usage: 272.4 KB


In [9]:
df_final.to_csv(
    "sui_factura_promedio_consolidado.csv",
    index=False,
    encoding="utf-8-sig"
)